In [0]:
from pyspark.sql.functions import col, lit
import datetime, json
from delta.tables import DeltaTable

# Define paths
silver_path = "abfss://silver@strdatabrickssadls.dfs.core.windows.net/supermarkets"
silver_table = "db_dataclassdev.silver.supermarkets"

# Read from bronze
df_bronze = spark.read.format("delta").table("db_dataclassdev.bronze.supermarkets")

# Transform
df_transformed = (
    df_bronze
    .withColumn("supermarket_No", col("supermarket_No").cast("int"))
    .withColumn("postal-code", col("postal-code").cast("int"))
    .withColumn("etl_record_created_date", lit(datetime.datetime.utcnow()))
    .withColumn("etl_record_modified_date", lit(datetime.datetime.utcnow()))
)

# Create silver table if not exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_table} (
    supermarket_No INT,
    `postal-code` INT,
    etl_record_created_date TIMESTAMP,
    etl_record_modified_date TIMESTAMP
)
USING DELTA
LOCATION '{silver_path}'
""")

# Perform Merge (Upsert)
target = DeltaTable.forPath(spark, silver_path)

(
    target.alias("target")
    .merge(
        df_transformed.alias("source"),
        "target.supermarket_No = source.supermarket_No"
    )
    .whenMatchedUpdate(set={
        "`postal-code`": "source.`postal-code`",
        "etl_record_modified_date": "source.etl_record_modified_date"
    })
    .whenNotMatchedInsertAll()
    .execute()
)

# Audit return to ADF
audit_info = {
    "fileName": "supermarkets",
    "rowCount": df_transformed.count(),
    "status": "Succeeded",
    "destinationPath": silver_path,
    "timestamp": str(datetime.datetime.utcnow())
}

dbutils.notebook.exit(json.dumps(audit_info))
